# Appendix C.1 — Consented Discharges to Controlled Waters with Conditions (PRO)

Evidence for every claim made about the PRO register in Appendix C of the demonstrator report.

Each section states the claim, then loads the source and prints **the actual records that cause it**.
Nothing here is asserted from memory: every number in the appendix is recomputed below.

**Sources used**

| file | what it is |
| --- | --- |
| `raw_datasets/access_database_csv_files/determinands.csv` | the conditions/limits register (national) |
| `raw_datasets/access_database_csv_files/consents_active.csv` | the consents register, in-force permits (national) |
| `raw_datasets/access_database_csv_files/consents_all_poole_harbour.csv` | a cut of *revoked* Poole Harbour permits |
| `ttl/regulation.ttl` | the delivered RDF graph, for claims about what was published |

Run top to bottom from anywhere inside the repository.


In [1]:
import os, json, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "raw_datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
RAW = ROOT / "raw_datasets"
REG = RAW / "access_database_csv_files"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
print("repository root:", ROOT)


repository root: /Users/waf/git/projects/demonstrator-poc


In [2]:
det = pd.read_csv(REG / "determinands.csv", dtype=str, low_memory=False)
sw = det[det.EA_REGION == "SW"]          # the demonstrator's region
print(f"determinands.csv: {len(det):,} rows nationally, {len(sw):,} in the South West")
det.head(3)


determinands.csv: 497,418 rows nationally, 80,024 in the South West


,EA_REGION,REGION,PERMIT_REF,VERSION,OUTLET_NUMBER,EFFLUENT_NUMBER,MONTH_FROM,MONTH_TO,CODE_1,VAL_1,CODE_2,VAL_2,CODE_3,VAL_3,DETE_CODE,UNITS,DETE,METHOD
0,AN,EA Anglian Region,AWCNF10417,1,1,2,01,12,MAXIMUM VALUE,9,MINIMUM VALUE,6,NaN,NaN,0061,PH UNITS,pH,ABSOLUTE
1,AN,EA Anglian Region,AWCNF10417,2,1,1,01,12,MAXIMUM VALUE,0.1,NaN,NaN,NaN,NaN,7080,MILLIGRAM PER LITRE,Chlorine : Total : In Situ as Cl2,ABSOLUTE
2,AN,EA Anglian Region,AWCNF10454,1,1,1,01,12,95 PERCENTILE,5,MAXIMUM VALUE,10,NaN,NaN,0111,MILLIGRAM PER LITRE,Ammoniacal Nitrogen as N,ABSOLUTE


---
## C.1.1 The object model is undocumented — the ABSOLUTE / COMPARATIVE distinction

> *"The absence of documentation on the object model and column definitions leaves the dataset open to
> interpretation — including on the ABSOLUTE/COMPARATIVE limit distinction, which materially affects how
> compliance is assessed."*

`METHOD` takes two values. Nothing shipped with the CSV says what either means, yet they cannot be
assessed the same way: an ABSOLUTE limit is judged against the discharge result alone, a COMPARATIVE one
against the *difference* between inlet and discharge. A consumer that does not know this will assess
1,354 South-West rules against the wrong quantity — and the rows look identical.


In [3]:
print(sw.METHOD.value_counts().to_string(), "\n")

comparative = sw[sw.METHOD == "COMPARATIVE"]
print(f"COMPARATIVE rules in the South West: {len(comparative):,}\n")
cols = ["PERMIT_REF","VERSION","OUTLET_NUMBER","EFFLUENT_NUMBER","DETE_CODE","DETE",
        "CODE_1","VAL_1","UNITS","METHOD"]
print("Rows that differ ONLY in METHOD -- same determinand, same units, same shape, different meaning:")
sample = pd.concat([
    comparative[(comparative.DETE_CODE == "0085") & (comparative.CODE_1 == "MAXIMUM VALUE")].head(2),
    sw[(sw.METHOD == "ABSOLUTE") & (sw.DETE_CODE == "0085") & (sw.CODE_1 == "MAXIMUM VALUE")].head(2),
])
sample[cols]


METHOD
ABSOLUTE       78670
COMPARATIVE     1354 

COMPARATIVE rules in the South West: 1,354

Rows that differ ONLY in METHOD -- same determinand, same units, same shape, different meaning:


,PERMIT_REF,VERSION,OUTLET_NUMBER,EFFLUENT_NUMBER,DETE_CODE,DETE,CODE_1,VAL_1,UNITS,METHOD
363438,040143,1,1,1,0085,BOD : 5 Day ATU,MAXIMUM VALUE,1.5,MILLIGRAM PER LITRE,COMPARATIVE
363546,040166,1,2,1,0085,BOD : 5 Day ATU,MAXIMUM VALUE,2,MILLIGRAM PER LITRE,COMPARATIVE
363430,011611,1,1,1,0085,BOD : 5 Day ATU,MAXIMUM VALUE,20,MILLIGRAM PER LITRE,ABSOLUTE
363434,011692,1,1,1,0085,BOD : 5 Day ATU,MAXIMUM VALUE,15,MILLIGRAM PER LITRE,ABSOLUTE


---
## C.1.2 A limit's statistic is load-bearing, and is carried positionally

> *"the register's `CODE_n`/`VAL_n` columns use no fixed order — permit `042451` puts the maximum in
> `CODE_1` and the percentile in `CODE_2`; permit `401747` does the reverse."*

Both rows below limit the same determinand (0111, ammonia) in the same units. Reading "the limit" out of
`VAL_1` gives 27 mg/l for one permit and 20 mg/l for the other — but for `042451` that 27 is the
upper-tier backstop, while its *binding* limit is the 7 mg/l 95th percentile sitting in `VAL_2`.


In [4]:
show = ["PERMIT_REF","VERSION","OUTLET_NUMBER","EFFLUENT_NUMBER","DETE_CODE","DETE",
        "CODE_1","VAL_1","CODE_2","VAL_2","UNITS","METHOD"]
pair = pd.concat([
    sw[(sw.PERMIT_REF == "042451") & (sw.DETE_CODE == "0111") & (sw.VERSION == "8")].head(1),
    sw[(sw.PERMIT_REF == "401747") & (sw.DETE_CODE == "0111") & (sw.VERSION == "3")].head(1),
])
pair[show]


,PERMIT_REF,VERSION,OUTLET_NUMBER,EFFLUENT_NUMBER,DETE_CODE,DETE,CODE_1,VAL_1,CODE_2,VAL_2,UNITS,METHOD
381907,042451,8,2b,1,0111,Ammoniacal Nitrogen as N,MAXIMUM VALUE,27,95 PERCENTILE,7,MILLIGRAM PER LITRE,ABSOLUTE
401337,401747,3,1,1,0111,Ammoniacal Nitrogen as N,95 PERCENTILE,20,MAXIMUM VALUE,48,MILLIGRAM PER LITRE,ABSOLUTE


In [5]:
# How often does the order flip? Count, across the region, which statistic lands in CODE_1.
order = (sw[sw.CODE_2.notna()]
         .groupby(["CODE_1", "CODE_2"]).size()
         .rename("rows").reset_index().sort_values("rows", ascending=False))
print("Statistic pairs, and which one occupies CODE_1:")
print(order.to_string(index=False))


Statistic pairs, and which one occupies CODE_1:
       CODE_1        CODE_2  rows
95 PERCENTILE MAXIMUM VALUE  3286
MINIMUM VALUE MAXIMUM VALUE  2867
MAXIMUM VALUE MINIMUM VALUE  1892
MAXIMUM VALUE 95 PERCENTILE   623
MAXIMUM VALUE    MEAN VALUE    20
   MEAN VALUE MAXIMUM VALUE     6
95 PERCENTILE 95 PERCENTILE     3
MINIMUM VALUE MINIMUM VALUE     2


Both orderings occur in real data, so *column position does not encode the statistic*. A consumer
must read `CODE_n` and pair it with `VAL_n`; taking `VAL_1` as "the limit" mis-states the obligation.

Nothing in the CSV distribution defines what `95 PERCENTILE` or `MEAN VALUE` require either — that a
percentile is assessed over a rolling 12-month sample set against a look-up table, and a mean against
the lower bound of a 90% confidence interval, lives in separate EA guidance. Read naively, a single
sample above the value looks like a breach.


---
## C.1.3 Grid references are published at three levels, and the coarsest is the one surfaced

> *"The register carries `DISCHARGE_NGR` (the discharge site), `OUTLET_GRID_REF` (the outlet) and
> `EFFLUENT_GRID_REF` (the effluent) … there is no such thing as a per-permit NGR."*

First, the three columns on one row — three different coordinates for one outlet:


In [6]:
act = pd.read_csv(REG / "consents_active.csv", dtype=str, low_memory=False)
print(f"consents_active.csv: {len(act):,} rows\n")
brockhill = act[act.DISCHARGE_SITE_NAME == "BROCKHILL WATERCRESS FARM"]
brockhill[["PERMIT_NUMBER","DISCHARGE_SITE_NAME","OUTLET_NUMBER","EFFLUENT_NUMBER",
           "DISCHARGE_NGR","OUTLET_GRID_REF","EFFLUENT_GRID_REF"]]


consents_active.csv: 72,176 rows



,PERMIT_NUMBER,DISCHARGE_SITE_NAME,OUTLET_NUMBER,EFFLUENT_NUMBER,DISCHARGE_NGR,OUTLET_GRID_REF,EFFLUENT_GRID_REF
53185,401058,BROCKHILL WATERCRESS FARM,1,1,SY8369092820,SY8370092950,SY8370092950
59067,401057,BROCKHILL WATERCRESS FARM,1,1,SY8369092820,SY8354092880,SY8354092880
60726,401057,BROCKHILL WATERCRESS FARM,2,1,SY8369092820,SY8343092870,SY8343092870
61563,401058,BROCKHILL WATERCRESS FARM,2,1,SY8369092820,SY8374092930,SY8374092930


`DISCHARGE_NGR` is identical on every row — it belongs to the **site**. The outlet and effluent refs
differ. Now the consequence: because the site reference is keyed on the permit, joining it on
`PERMIT_NUMBER` turns a site fact into a permit fact.


In [7]:
g = act[act.DISCHARGE_NGR.notna() & (act.DISCHARGE_NGR != "")]
per_ngr = g.groupby("DISCHARGE_NGR").agg(permits=("PERMIT_NUMBER", "nunique"),
                                         site_names=("DISCHARGE_SITE_NAME", "nunique"))
shared = per_ngr[per_ngr.permits > 1]
print(f"National grid references shared by more than one permit: {len(shared):,}")
print(f"  ... of which every permit names the SAME discharge site: {(shared.site_names == 1).sum():,}")
print("\nThe most-shared references:")
print(shared.sort_values("permits", ascending=False).head(5).to_string())


National grid references shared by more than one permit: 1,076
  ... of which every permit names the SAME discharge site: 1,074

The most-shared references:
               permits  site_names
DISCHARGE_NGR                     
SV0000000001        42           1
TF0973809347        24           1
SJ3160096700        18           1
TQ1020085700        16           1
SP2896807152        13           1


In [8]:
# The single most-shared reference is itself a defect worth reporting.
worst = shared.sort_values("permits", ascending=False).index[0]
print(f"Grid reference {worst} carries {shared.loc[worst, 'permits']} permits, all named:")
print(g[g.DISCHARGE_NGR == worst][["DISCHARGE_SITE_NAME"]].drop_duplicates().to_string(index=False))
print("\nA placeholder reference, decoding to the far south-west corner of the National Grid.")
print("\nThe most-shared reference naming a REAL site:")
real = shared[shared.index != worst].sort_values("permits", ascending=False).head(3)
for ref in real.index:
    names = g[g.DISCHARGE_NGR == ref].DISCHARGE_SITE_NAME.unique()
    print(f"  {ref}: {real.loc[ref,'permits']} permits -> {list(names)}")


Grid reference SV0000000001 carries 42 permits, all named:
DISCHARGE_SITE_NAME
   YWS UNKNOWN SITE

A placeholder reference, decoding to the far south-west corner of the National Grid.

The most-shared reference naming a REAL site:
  TF0973809347: 24 permits -> ['TALLINGTON LAKES LTD COMPLEX']
  SJ3160096700: 18 permits -> ['MERSEY DOCKS & HARBOUR CO']
  TQ1020085700: 16 permits -> ['RAF NORTHOLT']


### What that does to the delivered graph

The demonstrator publishes the site reference as the discharge point's default geometry, because that is
what the public register surfaces as "the" discharge location. The result, counted from the delivered
RDF rather than asserted:


In [9]:
import pyoxigraph as ox

store = ox.Store()
store.bulk_load(path=str(ROOT / "ttl" / "regulation.ttl"), format=ox.RdfFormat.TURTLE)

P = """PREFIX reg: <http://environment.data.gov.uk/ontology/regulation/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX wr:  <http://example.com/water-regulation/>
PREFIX gl:  <http://example.com/water-regulation/grid-level/>
PREFIX water: <http://environment.data.gov.uk/ontology/water/> """

def sparql(q, cols):
    return pd.DataFrame([[None if v is None else v.value for v in r] for r in store.query(P + q)],
                        columns=cols)

print(sparql("""SELECT (COUNT(DISTINCT ?d) AS ?discharge_points) WHERE { ?d a reg:DischargePoint }""",
             ["discharge_points"]).to_string(index=False))
print(sparql("""SELECT (COUNT(DISTINCT ?d) AS ?with_site_geometry) (COUNT(DISTINCT ?w) AS ?distinct_coordinates)
WHERE { ?d a reg:DischargePoint ; geo:hasGeometry ?g .
        ?g geo:asWKT ?w ; wr:gridReferenceLevel gl:site .
        FILTER(CONTAINS(STR(?w), "27700")) }""",
             ["with_site_geometry", "distinct_coordinates"]).to_string(index=False))


discharge_points
             122
with_site_geometry distinct_coordinates
               118                   38


In [10]:
print("Outlets per shared coordinate (site-level geometry), delivered graph:")
df = sparql("""SELECT ?w (COUNT(DISTINCT ?d) AS ?outlets) (GROUP_CONCAT(DISTINCT ?ref; separator=", ") AS ?permits)
WHERE { ?d a reg:DischargePoint ; geo:hasGeometry ?g .
        ?g geo:asWKT ?w ; wr:gridReferenceLevel gl:site .
        FILTER(CONTAINS(STR(?w), "27700"))
        BIND(REPLACE(STR(?d), "^.*/permit/([^/]*)/.*$", "$1") AS ?ref) }
GROUP BY ?w ORDER BY DESC(COUNT(DISTINCT ?d)) LIMIT 6""", ["wkt", "outlets", "permits"])
df["wkt"] = df.wkt.str.replace(r"^<[^>]*>\s*", "", regex=True)
df


Outlets per shared coordinate (site-level geometry), delivered graph:


,wkt,outlets,permits
0,POINT(389950 93350),16,042451
1,POINT(370900 90230),7,"401050, 041639"
2,POINT(383690 92820),7,"401058, 401057, 043245, 043244"
3,POINT(374920 87270),6,"401056, 043241, 043238"
4,POINT(363480 96120),5,"401223, 043276"
5,POINT(379801 90850),4,"400114%2FCF%2F01, 043260"


**118 outlets on 38 distinct coordinates**, and 4 of the 122 discharge points carry no coordinate at
all. The top row is 16 outlets of one permit on a single point; the third is 7 outlets across 4 different
permits sharing one — the Brockhill Watercress Farm cluster shown above, where the EA in fact samples
those outlets at four separate points over 100 m apart.


In [11]:
print("The 4 discharge points published with NO coordinate:")
sparql("""SELECT ?d WHERE {
  ?d a reg:DischargePoint .
  FILTER NOT EXISTS { ?d geo:hasGeometry ?g . ?g wr:gridReferenceLevel gl:site } }""", ["discharge_point"])


The 4 discharge points published with NO coordinate:


,discharge_point
0,http://example.com/water-regulation/permit/043091/outlet/1/effluent/1
1,http://example.com/water-regulation/permit/043091/outlet/1/effluent/2
2,http://example.com/water-regulation/permit/043091/outlet/2/effluent/1
3,http://example.com/water-regulation/permit/043091/outlet/1/effluent/3


In [12]:
# Why do they have none? The permit is simply absent from the consents register -- both extracts.
allp = pd.read_csv(REG / "consents_all_poole_harbour.csv", dtype=str, low_memory=False)
print("permit 043091 rows in consents_active.csv:            ", (act.PERMIT_NUMBER == "043091").sum())
print("permit 043091 rows in the revoked-permit extract:     ", (allp.PERMIT_NUMBER == "043091").sum())
print("permit 043091 rows in determinands.csv (its limits):  ", (det.PERMIT_REF == "043091").sum())

eff = pd.read_csv(REG / "effluents.csv", dtype=str, low_memory=False)
print("permit 043091 rows in effluents.csv (its outlets):    ", (eff.PERMIT_REF == "043091").sum())
print()
eff[eff.PERMIT_REF == "043091"][["PERMIT_REF","VERSION","OUTLET_NUMBER","EFFLUENT_NUMBER",
                                 "SPT_DESC","EFF_SAMPLE_POINT","DWF (m3_d)"]].sort_values(
    ["VERSION","OUTLET_NUMBER","EFFLUENT_NUMBER"]).head(8)


permit 043091 rows in consents_active.csv:             0
permit 043091 rows in the revoked-permit extract:      0
permit 043091 rows in determinands.csv (its limits):   66


permit 043091 rows in effluents.csv (its outlets):     14



,PERMIT_REF,VERSION,OUTLET_NUMBER,EFFLUENT_NUMBER,SPT_DESC,EFF_SAMPLE_POINT,DWF (m3_d)
200897,043091,1,1,1,SEWAGE DISCHARGES - FINAL/TREATED EFFLUENT - WATER COMPANY,50950709,47700
191529,043091,1,1,2,SEWAGE DISCHARGES - FINAL/TREATED EFFLUENT - WATER COMPANY,NaN,47700
186998,043091,1,1,3,SEWAGE DISCHARGES - STW STORM OVERFLOW/STORM TANK - WATER COMPANY,NaN,NaN
191727,043091,1,2,1,SEWAGE DISCHARGES - PUMPING STATION - WATER COMPANY,NaN,44940
186983,043091,2,1,1,SEWAGE DISCHARGES - FINAL/TREATED EFFLUENT - WATER COMPANY,NaN,47700
182612,043091,2,1,2,SEWAGE DISCHARGES - FINAL/TREATED EFFLUENT - WATER COMPANY,50950709,47700
191519,043091,2,1,3,SEWAGE DISCHARGES - STW STORM OVERFLOW/STORM TANK - WATER COMPANY,NaN,NaN
182613,043091,2,2,1,SEWAGE DISCHARGES - PUMPING STATION - WATER COMPANY,NaN,44940


This is not a small or marginal permit: it is a water-company sewage works with a dry-weather flow of
47,700 m³/day, fully described in `effluents.csv` and carrying limits in `determinands.csv`. It is
**absent from the consents register entirely**, so nothing published says where it discharges.

The registers therefore disagree about which permits exist. A consumer joining conditions to locations
loses this permit's four outlets silently — the join simply returns nothing, which reads the same as
"no outlets".


---
## C.1.4 Permit references are identifiers that look like numbers

> *"Most are 6-digit and zero-padded, but `400114/CF/01`, `EPRBB3593EG` and `EPRYP3399VF` are not."*


In [13]:
scoped = sparql("""SELECT ?p WHERE { ?p a water:WaterDischargePermit }""", ["permit"])
scoped["ref"] = (scoped.permit.str.rsplit("/", n=1).str[-1].str.replace("%2F", "/", regex=False))
print(f"permits in the catchment: {len(scoped)}")
print("\nnon-numeric references:")
print(scoped[~scoped.ref.str.isdigit()][["ref"]].to_string(index=False))
print("\nreferences that lose meaning if the column is read as a number:")
padded = scoped[scoped.ref.str.isdigit() & scoped.ref.str.startswith("0")]
print(f"  {len(padded)} of {len(scoped)} are zero-padded, e.g. {sorted(padded.ref)[:5]}")


permits in the catchment: 61

non-numeric references:
         ref
 EPRBB3593EG
400114/CF/01
 EPRYP3399VF

references that lose meaning if the column is read as a number:
  41 of 61 are zero-padded, e.g. ['040015', '040067', '040070', '040091', '040096']


In [14]:
# The same identifiers in WINEP -- unpadded. This is why the two datasets do not join as published.
import openpyxl
wb = openpyxl.load_workbook(RAW / "PR24 WINEP National Dataset.xlsx", read_only=True, data_only=True)
ws = wb["PR24 WINEP National Data"]
rows = ws.iter_rows(values_only=True)
H = list(next(rows))
winep = pd.DataFrame(list(rows), columns=H)
wq = winep[(winep.Water_Company == "Wessex Water Service Ltd") & (winep.EA_Function == "Water Quality")]
ids = wq.Licence_Permit_Obstruction_ID.dropna().astype(str).str.strip()
print("WINEP permit identifiers (sample):", sorted(ids.unique())[:8])
print("\nWINEP writes 42451; the PRO register writes 042451. Neither is wrong; nothing says they are the same.")
print("direct string join of WINEP ids onto PRO permit refs, matches:",
      ids.isin(set(scoped.ref)).sum(), "of", len(ids))
print("after zero-padding to 6 digits, matches:",
      ids.apply(lambda r: r.zfill(6) if r.isdigit() else r).isin(set(scoped.ref)).sum(), "of", len(ids))


WINEP permit identifiers (sample): ['', '010039', '010080', '010238', '010265', '010528', '010772', '010883']

WINEP writes 42451; the PRO register writes 042451. Neither is wrong; nothing says they are the same.
direct string join of WINEP ids onto PRO permit refs, matches: 45 of 1233
after zero-padding to 6 digits, matches: 59 of 1233


---
## C.1.5 Permit version effective dates are not published with the register

> *"the dates exist only on the public register web pages, which had to be fetched separately … the URL
> scheme there accommodates numeric refs only."*

Deciding which version of a permit applied on the day a sample was taken is fundamental to any
compliance statement. `determinands.csv` — the file that carries the limits — has no date column at all:


In [15]:
print("determinands.csv columns:")
print(list(det.columns))
print("\nany column mentioning a date:", [c for c in det.columns if "DATE" in c.upper()] or "none")
print("\nVersions of permit 042451 that carry limits:", sorted(sw[sw.PERMIT_REF == "042451"].VERSION.unique()))


determinands.csv columns:
['EA_REGION', 'REGION', 'PERMIT_REF', 'VERSION', 'OUTLET_NUMBER', 'EFFLUENT_NUMBER', 'MONTH_FROM', 'MONTH_TO', 'CODE_1', 'VAL_1', 'CODE_2', 'VAL_2', 'CODE_3', 'VAL_3', 'DETE_CODE', 'UNITS', 'DETE', 'METHOD']

any column mentioning a date: none

Versions of permit 042451 that carry limits: ['1', '2', '3', '4', '5', '6', '7', '8']


In [16]:
# The consents register has date columns, but only for the version rows it carries.
print("consents_active.csv, permit 042451 -- versions present and their dates:")
print(act[act.PERMIT_NUMBER == "042451"][
    ["PERMIT_NUMBER","PERMIT_VERSION","EFFECTIVE_DATE","REVOCATION_DATE","STATUS_DESCRIPTION"]
].drop_duplicates().to_string(index=False))
print("\n-> the register publishes the CURRENT version only; the 8 versions that carry limits are undated here.")


consents_active.csv, permit 042451 -- versions present and their dates:
PERMIT_NUMBER PERMIT_VERSION    EFFECTIVE_DATE REVOCATION_DATE    STATUS_DESCRIPTION
       042451              8 03/31/26 00:00:00             NaN VARIED UNDER EPR 2010

-> the register publishes the CURRENT version only; the 8 versions that carry limits are undated here.


In [17]:
# What had to be scraped from the public register instead, and what could not be.
dates = pd.read_csv(ROOT / "ttl" / "regulation" / "permit_version_dates.csv", dtype=str)
dated = dates[dates.effective_date.notna()]
print(f"version rows recovered from the public register: {len(dates)}")
print(f"  ... of which actually carry a date:            {len(dated)}")
print(f"  ... recovered but still undated:               {len(dates) - len(dated)}")
versions = sparql("""SELECT ?v WHERE { ?v a reg:PermitDocument }""", ["version"])
print(f"permit versions in the delivered graph:        {len(versions)}")
versions["ref"] = versions.version.str.extract(r"/permit/([^/]+)/version/")[0].str.replace("%2F", "/", regex=False)
versions["ver"] = versions.version.str.extract(r"/version/(\d+)")[0]
merged = versions.merge(dated, left_on=["ref","ver"], right_on=["permit_ref","version"], how="left")
undated = merged[merged.effective_date.isna()]
print(f"undated:                                      {len(undated)}\n")
print("undated versions, by permit:")
print(undated.groupby("ref").size().rename("versions").to_string())


version rows recovered from the public register: 165
  ... of which actually carry a date:            144
  ... recovered but still undated:               21
permit versions in the delivered graph:        170
undated:                                      26

undated versions, by permit:
ref
040070          2
040091          1
040136          1
041350          2
041639          2
043091          4
043238          1
043241          1
043244          1
043245          1
043246          1
043259          1
043276          1
043478          1
051340          1
400114/CF/01    2
EPRBB3593EG     2
EPRYP3399VF     1


The three non-numeric permits are undatable in principle: the public register's version URLs follow
`SW-{permit:06d}-{version:03d}`, which no `EPR…` reference fits. Their limits therefore cannot be tied
to a period at all, from any published source.


---
## C.1.6 Nitrogen carries two determinand codes

> *"The permit register codes Total Nitrogen `9686`; WINEP codes it `9194`. The EA determinand codelist
> gives both the label 'Nitrogen, Total as N'."*


In [18]:
codelist = json.loads((RAW / "determinand_codelist.json").read_text())
items = codelist["items"] if isinstance(codelist, dict) and "items" in codelist else codelist
flat = pd.json_normalize(items)
label_col = next(c for c in flat.columns if "label" in c.lower())
note_col = next(c for c in flat.columns if "notation" in c.lower())
both = flat[flat[note_col].astype(str).str.zfill(4).isin(["9686", "9194"])]
both[[note_col, label_col]]


,notation,prefLabel
7271,9194,"Nitrogen, Total as N"
7763,9686,"Nitrogen, Total as N"


In [19]:
print("PRO register, which nitrogen code does it use?")
print(sw[sw.DETE_CODE.isin(["9686", "9194"])].DETE_CODE.value_counts().to_string())
print("\nWINEP, which code does its nitrogen column map to? (from the shredder's column map)")
print("  Proposed_permit_other -> 9194, per ttl/winep/winep_to_db.py")
print("\nSo a query joining current to proposed nitrogen limits on the determinand code returns nothing,")
print("unless an explicit equivalence is asserted. The demonstrator asserts skos:exactMatch between them.")


PRO register, which nitrogen code does it use?
DETE_CODE
9686    18

WINEP, which code does its nitrogen column map to? (from the shredder's column map)
  Proposed_permit_other -> 9194, per ttl/winep/winep_to_db.py

So a query joining current to proposed nitrogen limits on the determinand code returns nothing,
unless an explicit equivalence is asserted. The demonstrator asserts skos:exactMatch between them.


In [20]:
print("The consequence the equivalence unlocks -- Poole WRC (401354), current vs proposed nitrogen:")
print(sw[(sw.PERMIT_REF == "401354") & (sw.DETE_CODE == "9686")][
    ["PERMIT_REF","VERSION","DETE_CODE","DETE","CODE_1","VAL_1","UNITS"]].sort_values("VERSION").tail(3).to_string(index=False))
print()
print(wq[wq.Licence_Permit_Obstruction_ID.astype(str).str.zfill(6) == "401354"][
    ["Action_ID","Action_Name","Proposed_permit_other"]].dropna(subset=["Proposed_permit_other"]).to_string(index=False))


The consequence the equivalence unlocks -- Poole WRC (401354), current vs proposed nitrogen:
PERMIT_REF VERSION DETE_CODE                 DETE     CODE_1 VAL_1               UNITS
    401354       7      9686 Nitrogen, Total as N MEAN VALUE    10 MILLIGRAM PER LITRE
    401354       8      9686 Nitrogen, Total as N MEAN VALUE    10 MILLIGRAM PER LITRE
    401354       9      9686 Nitrogen, Total as N MEAN VALUE    10 MILLIGRAM PER LITRE

 Action_ID                               Action_Name Proposed_permit_other
08WW100770                 Poole WRC Flow Monitoring                      
08WW100543             Poole WRC Overflow Monitoring                      
08WW102107 Poole WRC - Phosphorus & Nitrogen Removal               N 5mg/l
08WW101000     POOLE E STW SO Investigation (13242S)                      


---
## Summary of figures in Appendix C.1

| claim | value computed here |
| --- | --- |
| COMPARATIVE rules excluded (South West) | see C.1.1 |
| grid references shared by >1 permit (national) | see C.1.3 |
| mapped outlets / distinct coordinates / no coordinate | see C.1.3 |
| non-numeric permit references | see C.1.4 |
| permit versions left undated | see C.1.5 |
